# TP2 · Decide, evaluate, then discover language models

*Data Matching course · M2 MIAGE · Session 2 of 3 · Student version*

| Session | Notebook | Topic |
|---|---|---|
| 1 | TP1 | Know the data, standardise it, build and compare candidate pairs |
| **2** | **TP2** | **Decide and evaluate; discover language models** |
| 3 | TP3 | Language models in the matching pipeline |

This session alternates short lectures and practice. Each part of this notebook follows one block of the lecture:

| Part | Lecture block | Practice |
|---|---|---|
| 1 | Decision: rules and weights | decide with a rule and with a weighted sum (≈ 15 min) |
| 2 | Evaluation | precision, recall, F1, threshold (≈ 20 min) |
| 3 | Learning the weights, grey zone | logistic regression, grey zone (≈ 20 min) |
| 4 | Language models: tokens and vectors | tokens, word vectors, neighbours of a record (≈ 15 min) |
| 5 | Language models: prompt and limits | first questions to a local model (≈ 20 min) |

**Before the session**: install Ollama (https://ollama.com) and run `ollama pull qwen3.5:2b` in a terminal (about 2.7 GB). Part 5 needs it.

## 0. Set-up: rebuild the pipeline of TP1

The next cells repeat TP1 in a compact form: standardisation, multi-pass pairing and comparison. Run them; everything was explained in TP1.

In [3]:
#!pip install recordlinkage scikit-learn matplotlib sentence-transformers ollama pydantic
# On Linux, install the CPU-only PyTorch first (much smaller download):
#!pip install torch --index-url https://download.pytorch.org/whl/cpu
import re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import recordlinkage
from recordlinkage.datasets import load_febrl2
from recordlinkage.index import Block, SortedNeighbourhood
from sklearn.linear_model import LogisticRegression
SEED = 42
results = []     # one row per matching method (final table)

In [4]:
# Standardisation (TP1, Part 2). As discussed in TP1 (Exercise 2), we remove ALL frequent street types.
df_raw, true_links = load_febrl2(return_links=True)
ATTRIBUTES = list(df_raw.columns)
last_words = df_raw["address_1"].str.upper().str.split().str[-1].value_counts()
STREET_TYPES = last_words[last_words >= 20].index.tolist()

def clean(data):
    out = data.copy()
    for col in ATTRIBUTES:
        s = out[col].str.upper()
        s = s.str.replace(r"[^\w\s]", " ", regex=True)
        s = s.str.replace(r"\s+", " ", regex=True).str.strip()
        out[col] = s.replace("", np.nan)
    out["postcode"] = out["postcode"].str.replace(r"\D", "", regex=True).replace("", np.nan)
    pattern = r"\b(" + "|".join(map(re.escape, STREET_TYPES)) + r")\b"
    s = out["address_1"].str.replace(pattern, " ", regex=True)
    out["address_1_clean"] = s.str.replace(r"\s+", " ", regex=True).str.strip().replace("", np.nan)
    return out

df = clean(df_raw)
N = len(df)

In [5]:
# Tools: pair numbers, evaluation, choice of a threshold, display of a pair
def pair_keys(pairs):
    # One number per pair: the same number for (a, b) and (b, a).
    a = df.index.get_indexer(pairs.get_level_values(0))
    b = df.index.get_indexer(pairs.get_level_values(1))
    return np.minimum(a, b).astype(np.int64) * N + np.maximum(a, b)

def make_pairs(keys):
    keys = np.unique(keys)
    ids = df.index.to_numpy()
    return pd.MultiIndex.from_arrays([ids[keys % N], ids[keys // N]], names=["rec_id_1", "rec_id_2"])

TRUE_ALL = np.unique(pair_keys(true_links))

def evaluate(pred_pairs, true_keys, name, record=True):
    # Precision, recall and F1 of a set of predicted matches.
    pred = np.unique(pair_keys(pred_pairs))
    tp = int(np.isin(pred, true_keys).sum())
    fp, fn = len(pred) - tp, len(true_keys) - tp
    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    row = {"method": name, "TP": tp, "FP": fp, "FN": fn,
           "precision": round(precision, 4), "recall": round(recall, 4), "F1": round(f1, 4)}
    if record:
        results.append(row)
    return row

def best_threshold(scores, labels):
    # The threshold that gives the best F1 on (scores, labels). Rule: match if score >= threshold.
    scores, labels = np.asarray(scores, dtype=float), np.asarray(labels, dtype=bool)
    order = np.argsort(-scores)
    s, y_sorted = scores[order], labels[order]
    tp, fp = np.cumsum(y_sorted), np.cumsum(~y_sorted)
    precision, recall = tp / (tp + fp), tp / max(y_sorted.sum(), 1)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
    f1 = np.where(np.r_[s[1:] != s[:-1], True], f1, -1)
    i = int(np.argmax(f1))
    return s[i], f1[i]

def show_pair(a, b):
    return df.loc[[a, b], ATTRIBUTES].T

In [6]:
# Multi-pass pairing (TP1, Part 3) and comparison vectors (TP1, Part 4)
passes = [SortedNeighbourhood(left_on="surname", window=5), SortedNeighbourhood(left_on="given_name", window=5),
          Block(left_on="postcode"), Block(left_on="date_of_birth"), Block(left_on="soc_sec_id")]
candidates = None
for algorithm in passes:
    p = algorithm.index(df)
    candidates = p if candidates is None else candidates.union(p)
candidates = make_pairs(pair_keys(candidates))     # one row per pair, whatever the order (a, b) or (b, a)

cmp = recordlinkage.Compare()
for col, method in [("given_name", "jarowinkler"), ("surname", "jarowinkler"), ("street_number", "levenshtein"),
                    ("address_1_clean", "jarowinkler"), ("address_2", "jarowinkler"), ("suburb", "jarowinkler"),
                    ("postcode", "levenshtein"), ("date_of_birth", "levenshtein"), ("soc_sec_id", "levenshtein")]:
    cmp.string(col, col, method=method, label=col.replace("_clean", ""))
cmp.exact("state", "state", label="state")
features = cmp.compute(candidates, df)
y = np.isin(pair_keys(features.index), TRUE_ALL)
print(f"{len(features):,} candidate pairs, {y.sum()} true pairs among them")
features.head()

183,773 candidate pairs, 1934 true pairs among them


,,given_name,surname,street_number,address_1,address_2,suburb,postcode,date_of_birth,soc_sec_id,state
rec_id_1,rec_id_2,,,,,,,,,,
rec-1050-org,rec-2778-org,1.000000,0.466667,0.0,0.527778,0.544444,0.449495,0.75,0.500,0.000000,0
rec-2254-dup-2,rec-2778-org,1.000000,0.000000,0.0,0.495726,0.539869,0.388889,0.00,0.375,0.000000,0
rec-3251-org,rec-2778-org,0.000000,0.577778,0.0,0.430556,0.533333,0.576389,0.00,0.375,0.142857,1
rec-1942-org,rec-2778-org,0.540741,0.760000,0.0,0.505051,0.565079,0.527778,0.25,0.375,0.000000,0
rec-3977-org,rec-2778-org,0.447619,0.760000,0.0,0.666667,0.447222,0.408730,0.25,0.500,0.000000,0


## 1. Decision without learning *(lecture block "Décision")*

<a id='coeff'></a>
### 1.1 The rule of the TP: count the agreements

We turn each score into an **agreement** (1 if the score is high enough, 0 otherwise) and we count the agreements. As in the original TP, we declare a match with at least 8 agreements out of 9 (the street number is not used). This is a compact **truth table**.

In [7]:
agree = (features >= 0.85).astype(int)
agree["soc_sec_id"] = (features["soc_sec_id"] >= 0.6).astype(int)
agree["postcode"] = (features["postcode"] == 1).astype(int)
agree = agree.drop(columns="street_number")
n_agree = agree.sum(axis=1)
rule_pred = features.index[(n_agree > 7).to_numpy()]
print(n_agree.value_counts().sort_index(ascending=False))
print(f"\nThe rule declares {len(rule_pred)} matches")

9       437
8       664
7       498
6       234
5        85
4        16
3       320
2     29780
1    114072
0     37667
Name: count, dtype: int64

The rule declares 1101 matches


### 1.2 A weighted sum: some attributes count more (coefficient-based technique)

An agreement on *soc_sec_id* is stronger evidence than an agreement on *state*. We give each attribute a weight chosen **by hand**, and we compute the weighted average of the scores (between 0 and 1). We declare a match above a threshold, also chosen by hand.

In [8]:
WEIGHTS = {"given_name": 1.5, "surname": 1.5, "street_number": 1, "address_1": 1, "address_2": 0.5,
           "suburb": 1, "postcode": 1, "state": 0.5, "date_of_birth": 2, "soc_sec_id": 3}
w = pd.Series(WEIGHTS)[features.columns]
wscore = (features * w).sum(axis=1) / w.sum()      # weighted average of the scores
THRESHOLD = 0.8
wsum_pred = features.index[(wscore >= THRESHOLD).to_numpy()]
print(f"The weighted sum declares {len(wsum_pred)} matches")

The weighted sum declares 1597 matches


### Exercise 1 · Look at the pairs near the threshold

<font color='red'>

***To do***: display the 2 pairs with the **lowest** weighted score **above** the threshold, and the 2 pairs with the **highest** score **below** it (use `show_pair`). Would you declare them as matches? What does it tell you about the choice of the threshold?

</font>

In [ ]:
# Your code here

*Your answer:*

...

## 2. Evaluation *(lecture block "Évaluation")*

We compare the predicted matches with the true pairs. Three counts:
* **TP**: predicted matches that are true pairs;
* **FP**: predicted matches that are not true pairs (two persons merged);
* **FN**: true pairs that we did not predict (duplicates left), including the true pairs lost at the pairing step.

Three questions:
* **Precision** = TP / (TP + FP): *among the pairs that I declared, how many are true?*
* **Recall** = TP / (TP + FN): *among all true pairs, how many did I find?*
* **F1**: *one single score, high only if both are high.*

<img src="img/image3.png" width="45%"/>

<font color='red'>

***To do (from the original TP)***: compute TP, FP, FN, precision, recall and F1 of the rule (1.1) by yourself. Hint: `pair_keys(rule_pred)` gives the numbers of the predicted pairs; `np.isin(a, b)` tells which elements of `a` are in `b`. Then check your result with the function `evaluate`.

</font>

In [ ]:
# Your code here

In [ ]:
print(evaluate(rule_pred, TRUE_ALL, "rule", record=False))
print(evaluate(wsum_pred, TRUE_ALL, "weighted sum", record=False))
print(f"Accuracy of a lazy method that always answers 'non-match': {1 - y.mean():.2%} (and recall = 0)")

The last line shows the **accuracy trap**: almost all candidate pairs are non-matches, so a method that finds nothing still looks "99% correct".

**The threshold is a trade-off.** A low threshold finds more true pairs (recall ↑) but makes more mistakes (precision ↓).

In [ ]:
curve = []
for t in np.arange(0.60, 0.96, 0.05):
    row = evaluate(features.index[(wscore >= t).to_numpy()], TRUE_ALL, f"t={t:.2f}", record=False)
    curve.append({"threshold": round(t, 2), "precision": row["precision"], "recall": row["recall"], "F1": row["F1"]})
curve = pd.DataFrame(curve).set_index("threshold")
curve.plot(marker="o", grid=True, title="Weighted sum: effect of the threshold")
plt.show()
curve

### Exercise 2 · Move the threshold of the rule

<font color='red'>

***To do***: evaluate the rule with more than 6, more than 7 and more than 8 agreements. How do precision and recall change? Which threshold would you choose for a mailing list? For medical records?

</font>

In [ ]:
# Your code here

*Your answer:*

...

## 3. Learning the weights *(lecture block "Apprendre les poids")*

### 3.1 Learn on one part, evaluate on another part

We split the **persons** into two halves: the *train* part to learn the weights and to choose the threshold, the *test* part to evaluate, and nothing else. We split by person, not by pair: otherwise, the duplicates of a person could be on both sides (*data leakage*).

In [ ]:
entity = pd.Series(df.index.str.extract(r"rec-(\d+)-", expand=False), index=df.index)
rng = np.random.default_rng(SEED)
all_entities = entity.unique()
in_test = entity.isin(set(rng.choice(all_entities, size=len(all_entities) // 2, replace=False)))

def split_masks(pairs):
    # Pairs with both records in the train part / in the test part. Mixed pairs are removed.
    a = in_test.reindex(pairs.get_level_values(0)).to_numpy(dtype=bool)
    b = in_test.reindex(pairs.get_level_values(1)).to_numpy(dtype=bool)
    return (~a & ~b), (a & b)

train_mask, test_mask = split_masks(features.index)
_, test_part = split_masks(true_links)
TRUE_TEST = np.unique(pair_keys(true_links[test_part]))
test_pairs = features.index[test_mask]
print(f"train: {train_mask.sum():,d} pairs | test: {test_mask.sum():,d} pairs ({len(TRUE_TEST)} true pairs)")

# For a fair comparison, the rule and the weighted sum are evaluated again on the test part only.
evaluate(features.index[test_mask & (n_agree > 7).to_numpy()], TRUE_TEST, "rule of the TP (8 agreements out of 9)")
evaluate(features.index[test_mask & (wscore >= THRESHOLD).to_numpy()], TRUE_TEST, "weighted sum, weights by hand")
pd.DataFrame(results).set_index("method")

### 3.2 The logistic regression learns the weights

The model looks at the train pairs and adjusts one weight per attribute, so that its probability is high for the true pairs and low for the other pairs. We choose the threshold on the train part and evaluate on the test part.

In [ ]:
logreg = LogisticRegression(max_iter=1000).fit(features[train_mask], y[train_mask])
t_lr, _ = best_threshold(logreg.predict_proba(features[train_mask])[:, 1], y[train_mask])
p_lr_test = logreg.predict_proba(features[test_mask])[:, 1]
print(f"threshold = {t_lr:.3f}")
print(evaluate(test_pairs[p_lr_test >= t_lr], TRUE_TEST, "logistic regression (weights learned)"))
learned = pd.Series(logreg.coef_[0], index=features.columns).sort_values()
learned.plot.barh(title="Weights learned by the logistic regression", grid=True)
plt.show()

### 3.3 Two thresholds and a grey zone

Probability < 0.05: **non-match**. Probability ≥ 0.95: **match**. In between: the **grey zone**, where the method hesitates.

In [ ]:
LOW, HIGH = 0.05, 0.95
zone = pd.cut(p_lr_test, [0, LOW, HIGH, 1.01], labels=["non-match", "grey zone", "match"], right=False)
grey = np.where((p_lr_test >= LOW) & (p_lr_test < HIGH))[0]
pd.crosstab(zone, np.where(y[test_mask], "true pair", "other pair"), margins=True)

The grey zone is small, but it mixes true pairs and other pairs. In session 3, a language model will decide these pairs.

**Summary of the classical methods (test part).** This table is the reference for session 3.

In [ ]:
pd.DataFrame(results).drop_duplicates("method", keep="last").set_index("method")

### Exercise 3 · Your weights against the learned weights

<font color='red'>

***To do***: compare the weights of `WEIGHTS` (by hand) with the learned weights (`learned`). Which attribute gets the largest learned weight? Which one gets a small weight? Explain with the idea "a rare agreement is strong evidence".

</font>

In [ ]:
# Your code here

*Your answer:*

...

### Exercise 4 · The grey zone, seen by a human

<font color='red'>

***To do***: display 2 pairs of the grey zone (the closest to 0.5). Decide yourself: match or non-match? What information did you use that the method does not have?

</font>

In [ ]:
# Your code here

*Your answer:*

...

## 4. Language models: tokens and vectors *(lecture block "Modèles de langage")*

We now discover the two kinds of models of session 3:

| Model | Family | What it does | Size |
|---|---|---|---|
| *all-MiniLM-L6-v2* | encoder | turns a text into a vector of 384 numbers | 22 million parameters, about 90 MB |
| *qwen3.5:2b* (Ollama) | decoder (generative) | writes an answer, token by token | about 2 billion parameters, about 2.7 GB |

Language models read **text**, not tables. We write each record as one short text (*serialisation*); missing values are skipped.

In [ ]:
EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
OLLAMA_MODEL = "qwen3.5:0.8b"      # or "qwen3.5:0.8b" (faster, less accurate)

def record_to_text(rec):
    return "; ".join(f"{col.replace('_', ' ')}: {rec[col]}" for col in ATTRIBUTES if pd.notna(rec[col]))

record_text = df_raw.apply(record_to_text, axis=1)     # raw values, in lower case
print(record_text.iloc[0])

from sentence_transformers import SentenceTransformer
encoder = SentenceTransformer(EMB_MODEL, device="cpu")
for text in ["street", "sarah", "bruhn", "brunh", "kellerberrin", "given name: sarah; surname: bruhn"]:
    print(f"{text!r:38} -> {encoder.tokenizer.tokenize(text)}")

**Tokens.** Frequent words are one token. Rare words are cut into pieces (`##` marks a piece that continues a word). A typo changes only some pieces.

**Vectors.** The encoder turns each text into 384 numbers. Two texts with a close meaning get close vectors. We measure the closeness with the **cosine**: 1 = same direction, close to 0 = unrelated.

In [ ]:
words = ["street", "road", "avenue", "monday", "tuesday", "sydney", "melbourne", "william", "bill"]
W = encoder.encode(words, normalize_embeddings=True)
pd.DataFrame(W @ W.T, index=words, columns=words).round(2)

**From records to vectors.** We encode the 5000 records (about half a minute), then we look for the **nearest neighbours** of one record. No blocking key is needed.

In [ ]:
t0 = time.time()
E = encoder.encode(record_text.tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=True)
print(f"{E.shape[0]} records x {E.shape[1]} numbers in {time.time() - t0:.0f} s")
person = df.index.str.extract(r"rec-(\d+)-", expand=False)

def nearest(rec_id, k=5):
    i = df.index.get_loc(rec_id)
    scores = E @ E[i]
    best = np.argsort(-scores)[1:k + 1]          # position 0 is the record itself
    out = df_raw.iloc[best][["given_name", "surname", "address_1", "suburb", "date_of_birth"]].copy()
    out.insert(0, "cosine", scores[best].round(3))
    out.insert(1, "same person?", person[best] == person[i])
    return out

example = df.index[person == person.value_counts().index[0]][0]
display(df_raw.loc[[example], ["given_name", "surname", "address_1", "suburb", "date_of_birth"]])
nearest(example)

### Exercise 5 · Similar is not identical

<font color='red'>

***To do***: find two pairs of words with a high cosine that do **not** name the same thing. Why is it a risk for the matching? Which measure of session 1 would not make this mistake?

</font>

In [ ]:
# Your code here

*Your answer:*

...

## 5. Language models: prompt, temperature, limits *(lecture block "Générer, prompt, limites")*

A generative model writes an answer, token by token. We send it a **prompt**. The function `ask` sends a prompt to the local model; the first call also loads the model in memory.

In [ ]:
import ollama

def ask(prompt, temperature=0.0, system=None):
    messages = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
    kwargs = dict(model=OLLAMA_MODEL, messages=messages, options={"temperature": temperature, "num_ctx": 2048})
    try:
        response = ollama.chat(think=False, **kwargs)    # no "thinking" step: faster on a CPU
    except (TypeError, ollama.ResponseError):
        response = ollama.chat(**kwargs)                 # old client, or model without thinking mode
    return response["message"]["content"].strip()

try:
    print(ask("In one sentence: what is data matching?"))
except Exception as error:
    print("Ollama is not ready:", error)
    print(f"Install Ollama (https://ollama.com), start it, then run in a terminal: ollama pull {OLLAMA_MODEL}")

**Temperature.** At 0, the model always takes the most probable token: the same prompt gives the same answer. Higher, the answers vary.

In [ ]:
prompt = "Give one possible nickname for the first name Elizabeth. Answer with one word only."
for t in [0.0, 0.0, 1.2, 1.2, 1.2]:
    print(f"temperature {t}: {ask(prompt, temperature=t)}")

**Knowledge learned from texts, and its limits.** The model knows that *Bill* is a nickname of *William*: Jaro–Winkler cannot know it. But it also answers questions that have no true answer: a **plausible** answer is not a **true** answer (*hallucination*).

In [ ]:
for q in ["Is 'Bill' a usual nickname of 'William'? Answer Yes or No.",
          "Do 'St Kilda' and 'Saint Kilda' name the same place? Answer Yes or No.",
          "In which year was Forbes Street in Kellerberrin (Australia) renamed? Answer with a year only."]:
    print(f"{q}\n  -> {ask(q)}")

### Exercise 6 · Zero-shot and few-shot prompts

<font color='red'>

***To do***: ask the model whether two first names can be the same person's name: *Peggy / Margaret*, *Jon / John*, *Jack / Jacqueline*. First without any example (*zero-shot*), then with two solved examples at the beginning of the prompt (*few-shot*): *Bill / William: Yes* and *Sarah / Sandra: No*. Do the answers change? Is their format more regular?

</font>

In [ ]:
# Your code here

*Your answer:*

...

## What's next?

In session 3, we put these two models into the pipeline: the encoder for the pairing and the comparison, the generative model for the standardisation of addresses and for the decision on the grey zone. The table of Part 3 is the reference.